# ROAD ADV ATTACKS
**Function to load subset and filter normal samples**\
**Adversarial Samples Generation**\
**Constraint Compliance**

In [3]:
def load_dataset(files):
    fn_flag = 1
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    df=df[df['Flag'] == fn_flag].copy()
    X = df.drop(columns=['Flag'])   # features only
    y = df['Flag'].values           
    X.columns = [c.replace("[", "_").replace("]", "").replace("<", "_") for c in X.columns]
    return X, y

def generate_constrained_attack(estimator, X, method="FGSM", eps=1.0, eps_step=0.1, max_iter=10):
    """
    Generate adversarial samples with IVN constraints using FGSM, BIM, or PGD.
    
    Parameters:
    - estimator: ART classifier
    - X: np.ndarray or pd.DataFrame, shape (n_samples, 10)
    - method: str, one of {"FGSM", "BIM", "PGD"}
    - eps: float, total allowed perturbation
    - eps_step: float, step size
    - max_iter: int, only used for iterative attacks (BIM, PGD)
    
    Returns:
    - X_adv: adversarial samples (np.ndarray), clipped to [0, 255] on DATA[0]-[7]
    """
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()

    # Only allow perturbation on DATA[0]-[7]
    perturbation_mask = np.array([False, False] + [True] * 8)

    # Select the attack method
    if method == "FGSM":
        attack = FastGradientMethod(estimator=estimator, eps=eps, eps_step=eps_step)
    elif method == "BIM":
        attack = BasicIterativeMethod(estimator=estimator, eps=eps, eps_step=eps_step, max_iter=max_iter, verbose=False)
    elif method == "PGD":
        attack = ProjectedGradientDescent(estimator=estimator, eps=eps, eps_step=eps_step, max_iter=max_iter, num_random_init=1, verbose=False)
    else:
        raise ValueError("Unsupported attack method. Choose 'FGSM', 'BIM', or 'PGD'.")

    # Generate adversarial examples
    X_adv = attack.generate(x=X, mask=perturbation_mask)

    # Clip DATA[0] to DATA[7] to valid byte range
    X_adv[:, 2:] = np.clip(X_adv[:, 2:], 0, 255)
    
    # Log attack details
    print(f"[{method}] Generated adversarial examples with eps={eps}, eps_step={eps_step}, max_iter={max_iter if method != 'FGSM' else 'N/A'}")
    
    return X_adv
def constraint_compliant(X):
    X_np = X.to_numpy() if isinstance(X, pd.DataFrame) else X
    return (
        (X_np[:, 0] >= 0) & (X_np[:, 0] <= 1068) &  # CAN ID
        (X_np[:, 1] >= 1) & (X_np[:, 1] <= 8) &     # DLC
        np.all((X_np[:, 2:] >= 0) & (X_np[:, 2:] <= 255), axis=1)  # DATA[0]-[7]
    )


In [4]:
def evaluate_model_on_adversarial_tabular(
    model,
    X_clean,
    y_clean,          # <-- now you pass true labels (0=normal, 1=attack)
    X_adv_dict,
    model_name="RF",
    attack_name="BIM",
    is_probabilistic=False
):
    results = []
    sample_size = len(X_clean)

    if is_probabilistic:
        y_prob = model.predict(X_clean).ravel()   # values in (0, 1)
        y_pred_clean = (y_prob >= 0.5).astype(int) #y_pred_clean = np.argmax(model.predict(X_clean), axis=1)
    else:
        y_pred_clean = model.predict(X_clean)


    # Confusion matrix (TN, FP, FN, TP)
    cm_clean = confusion_matrix(y_clean, y_pred_clean, labels=[0,1])
    tn, fp, fn, tp = cm_clean.ravel()
    f1_clean = f1_score(y_clean, y_pred_clean, average='weighted')

    for eps_val, X_adv in X_adv_dict.items():
        valid_mask = constraint_compliant(X_adv)
        X_adv_valid = X_adv[valid_mask]
        if len(X_adv_valid) == 0:
            continue

        # Ground truth is same as y_clean for aligned samples
        y_true_adv = y_clean[valid_mask]
        if is_probabilistic:
            y_prob_adv = model.predict(X_adv_valid).ravel()   # values in (0, 1)
            y_pred_adv = (y_prob_adv >= 0.5).astype(int)
            y_pred_adv_binary = (y_pred_adv.ravel() >= 0.5).astype(int)
        else:
            y_pred_adv = model.predict(X_adv_valid)
            y_pred_adv_binary = y_pred_adv.astype(int)


        f1_adv = f1_score(y_true_adv, y_pred_adv_binary, average='weighted')
        # fn_adv = np.sum(y_pred_adv_binary)
        # asr = fn_adv / len(X_adv_valid)
        fn_adv = np.sum(y_pred_adv_binary == 0)
        asr = fn_adv / len(y_pred_adv_binary)   # if ASR = misclassification rate



        results.append([
            model_name, sample_size, f"{f1_clean*100:.1f}%", fn, attack_name, eps_val,
            f"{f1_adv*100:.1f}%", fn_adv, f"{asr:.1%}"
        ])
    columns = ["Model", "Sample Size",
               "F1 Score", "FN", 
               "Attack", "Epsilon",
               "F1 Score (Adv)", "FN (Adv)", "ASR"]
    
    return pd.DataFrame(results, columns=columns)


In [5]:
def run_attacks(dataset_name, X_normal, y_clean, models, attack_methods):
    results = {}
    cols = X_normal.columns
    for attack in attack_methods:
        X_adv_dict_all = {}  # holds adversarial sets per model

        # === Generate adversarial examples ===
        if attack in ["FGSM", "BIM", "PGD"]:
            # These only use the DNN
            if attack == "FGSM":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="FGSM", eps=1.0, eps_step=0.1),
                    5: generate_constrained_attack(dnn_art, X_normal, method="FGSM", eps=5.0, eps_step=0.1)
                }
            elif attack == "BIM":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="BIM", eps=1.0, eps_step=0.1, max_iter=10),
                    5: generate_constrained_attack(dnn_art, X_normal, method="BIM", eps=5.0, eps_step=0.1, max_iter=10)
                }
            elif attack == "PGD":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="PGD", eps=1.0, eps_step=0.1, max_iter=20),
                    5: generate_constrained_attack(dnn_art, X_normal, method="PGD", eps=5.0, eps_step=0.1, max_iter=20)
                }

            # store once, reused by all models
            X_adv_dict_all = {"DNN": X_adv_dict}

        else:
            continue
    
        # === Evaluate across all models ===
        for model_name, model_obj in models.items():
            X_fixed = pd.DataFrame(X_normal.values, columns=cols)

            # choose the right adversarial examples
            if attack == "DT":
                X_adv_fixed = {eps: pd.DataFrame(X_adv, columns=cols)
                               for eps, X_adv in X_adv_dict_all[model_name].items()}
            else:
                X_adv_fixed = {eps: pd.DataFrame(X_adv, columns=cols)
                               for eps, X_adv in X_adv_dict_all["DNN"].items()}

            result_key = f"{dataset_name}_{model_name}_{attack}"
            results[result_key] = evaluate_model_on_adversarial_tabular(model_obj, X_fixed, y_clean, X_adv_fixed, 
                                model_name=model_name, attack_name=attack, is_probabilistic=(model_name == "DNN"))

    return results


# False Negatives (FNs)

In [6]:
import glob
import os
import pandas as pd
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from art.attacks.evasion import FastGradientMethod, BasicIterativeMethod, ProjectedGradientDescent
from tensorflow.keras.models import load_model
from art.estimators.classification import KerasClassifier, SklearnClassifier
from art.estimators.classification import XGBoostClassifier
from art.estimators.classification import TensorFlowV2Classifier
from tensorflow.keras.optimizers import legacy as legacy_optimizers
import tensorflow as tf
from sklearn.metrics import f1_score, confusion_matrix


csv_folder = 'preprocessed'  # Update this
csv_files = sorted(glob.glob(os.path.join(csv_folder, "*.csv")))
csa = csv_files[:3]
fa = csv_files[3:6]
mecta = csv_files[6:7]
msa = csv_files[7:10]
rloffa = csv_files[10:13]
rlona = csv_files[13:16]
print(csa, fa,mecta,msa,rloffa,rlona)

# Load datasets with labels This lable is for FALSE POSITIVES #fp_flag = 0
fn_flag = 1
X_csa, y_csa = load_dataset(csa)
X_fa, y_fa = load_dataset(fa)
X_mecta, y_mecta = load_dataset(mecta)
X_msa, y_msa = load_dataset(msa)
X_rloffa, y_rloffa = load_dataset(rloffa)
X_rlona, y_rlona = load_dataset(rlona)

# Load models
dnn_model = load_model("models/dnn_model.h5")
dt_model = joblib.load("models/dt_model.pkl")
rf_model = joblib.load("models/rf_model.pkl")
et_model = joblib.load("models/et_model.pkl")
xgb_model = joblib.load("models/xgboost_model.pkl")

#Wrap Model
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
dnn_art = TensorFlowV2Classifier(
    model=dnn_model,
    loss_object=loss_object,
    optimizer=legacy_optimizers.Adam(),
    nb_classes=2,
    input_shape=(X_csa.shape[1],),
    clip_values=(0, 255),
)


models = {"DNN": dnn_art, "DT": dt_model, "RF": rf_model, "ET": et_model, "XGBoost": xgb_model}
datasets = {"rlona": (X_rlona, y_rlona), "rloffa": (X_rloffa, y_rloffa), "msa": (X_msa, y_msa), "mecta": (X_mecta, y_mecta), "fa": (X_fa, y_fa), "csa": (X_csa,y_csa)}
attack_methods = ["FGSM", "BIM", "PGD"]

# mapping dataset keys to names
dataset_labels = {
    "csa": "Correlated Signal Attack",
    "fa": "Fuzzing Attack",
    "mecta": "Max Engine Coolant Temp Attack",
    "msa": "max_speedometer_attack",  
    "rloffa": "Reverse Light Off Attack",
    "rlona": "Reverse Light On Attack"
}

# Run attacks for all datasets
all_results_fn= {}
for dataset_key, (X,y) in datasets.items():
    dataset_results = run_attacks(dataset_key, X, y, models, attack_methods)

    # merge all model/attack results into one big DataFrame for this dataset
    merged_df = pd.concat(dataset_results.values(), ignore_index=True)

    # store under dataset key
    all_results_fn[dataset_key] = merged_df

# print(dnn_model.output_shape)


/opt/miniconda3/envs/road/lib/python3.10/site-packages/art/estimators/certification/__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


['preprocessed/csa1.csv', 'preprocessed/csa2.csv', 'preprocessed/csa3.csv'] ['preprocessed/fa1.csv', 'preprocessed/fa2.csv', 'preprocessed/fa3.csv'] ['preprocessed/mecta.csv'] ['preprocessed/msa1.csv', 'preprocessed/msa2.csv', 'preprocessed/msa3.csv'] ['preprocessed/rloffa1.csv', 'preprocessed/rloffa2.csv', 'preprocessed/rloffa3.csv'] ['preprocessed/rlona1.csv', 'preprocessed/rlona2.csv', 'preprocessed/rlona3.csv']
Metal device set to: Apple M4

systemMemory: 16.00 GB
maxCacheSize: 5.92 GB



2026-01-29 19:13:39.887492: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=N/A
[FGSM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=N/A
[BIM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=10
[BIM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=10
[PGD] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=20
[PGD] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=20
[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=N/A
[FGSM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=N/A
[BIM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=10
[BIM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=10
[PGD] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=20
[PGD] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=20
[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, ma

In [7]:
for dataset_key, df in all_results_fn.items():
    dataset_label = dataset_labels.get(dataset_key, dataset_key)
    print(dataset_label)
    print(f"\n=== {dataset_label} ===")
    display(df)   # Jupyter; if terminal: print(df.to_string())

Reverse Light On Attack

=== Reverse Light On Attack ===


,Model,Sample Size,F1 Score,FN,Attack,Epsilon,F1 Score (Adv),FN (Adv),ASR
0,DNN,15195,94.2%,1674,FGSM,1,94.2%,1665,11.0%
1,DNN,15195,94.2%,1674,FGSM,5,94.2%,1665,11.0%
2,DT,15195,99.1%,273,FGSM,1,99.1%,273,1.8%
3,DT,15195,99.1%,273,FGSM,5,99.1%,273,1.8%
4,RF,15195,99.1%,258,FGSM,1,99.1%,258,1.7%
5,RF,15195,99.1%,258,FGSM,5,99.1%,258,1.7%
6,ET,15195,99.1%,273,FGSM,1,99.1%,273,1.8%
7,ET,15195,99.1%,273,FGSM,5,99.1%,273,1.8%
8,XGBoost,15195,98.9%,324,FGSM,1,98.9%,324,2.1%
9,XGBoost,15195,98.9%,324,FGSM,5,98.9%,324,2.1%


Reverse Light Off Attack

=== Reverse Light Off Attack ===


,Model,Sample Size,F1 Score,FN,Attack,Epsilon,F1 Score (Adv),FN (Adv),ASR
0,DNN,10605,54.0%,6687,FGSM,1,44.4%,4566,71.5%
1,DNN,10605,54.0%,6687,FGSM,5,44.4%,4566,71.5%
2,DT,10605,72.4%,4590,FGSM,1,71.9%,2799,43.8%
3,DT,10605,72.4%,4590,FGSM,5,71.9%,2799,43.8%
4,RF,10605,71.9%,4650,FGSM,1,71.2%,2859,44.8%
5,RF,10605,71.9%,4650,FGSM,5,71.2%,2859,44.8%
6,ET,10605,72.4%,4590,FGSM,1,71.9%,2799,43.8%
7,ET,10605,72.4%,4590,FGSM,5,71.9%,2799,43.8%
8,XGBoost,10605,72.4%,4593,FGSM,1,71.9%,2799,43.8%
9,XGBoost,10605,72.4%,4593,FGSM,5,71.9%,2799,43.8%


max_speedometer_attack

=== max_speedometer_attack ===


...

,Model,Sample Size,F1 Score,FN,Attack,Epsilon,F1 Score (Adv),FN (Adv),ASR
0,DNN,88,0.0%,88,FGSM,1,0.0%,6,100.0%
1,DNN,88,0.0%,88,FGSM,5,0.0%,6,100.0%
2,DT,88,82.7%,26,FGSM,1,50.0%,4,66.7%
3,DT,88,82.7%,26,FGSM,5,50.0%,4,66.7%
4,RF,88,81.9%,27,FGSM,1,50.0%,4,66.7%
5,RF,88,81.9%,27,FGSM,5,50.0%,4,66.7%
6,ET,88,81.9%,27,FGSM,1,50.0%,4,66.7%
7,ET,88,81.9%,27,FGSM,5,50.0%,4,66.7%
8,XGBoost,88,81.9%,27,FGSM,1,50.0%,4,66.7%
9,XGBoost,88,81.9%,27,FGSM,5,50.0%,4,66.7%


Fuzzing Attack

=== Fuzzing Attack ===


,Model,Sample Size,F1 Score,FN,Attack,Epsilon,F1 Score (Adv),FN (Adv),ASR
0,DNN,1061,98.9%,24,FGSM,1,98.0%,24,3.8%
1,DNN,1061,98.9%,24,FGSM,5,98.0%,24,3.8%
2,DT,1061,100.0%,0,FGSM,1,100.0%,0,0.0%
3,DT,1061,100.0%,0,FGSM,5,100.0%,0,0.0%
4,RF,1061,100.0%,0,FGSM,1,100.0%,0,0.0%
5,RF,1061,100.0%,0,FGSM,5,100.0%,0,0.0%
6,ET,1061,100.0%,0,FGSM,1,100.0%,0,0.0%
7,ET,1061,100.0%,0,FGSM,5,100.0%,0,0.0%
8,XGBoost,1061,100.0%,0,FGSM,1,100.0%,0,0.0%
9,XGBoost,1061,100.0%,0,FGSM,5,100.0%,0,0.0%


Correlated Signal Attack

=== Correlated Signal Attack ===


,Model,Sample Size,F1 Score,FN,Attack,Epsilon,F1 Score (Adv),FN (Adv),ASR


In [11]:
# import matplotlib.pyplot as plt

# for key, df in all_results_fn.items():
#     df.to_csv(f"results/{key}_fn.csv", index=False)

for key, df in all_results_fn.items():
    df = df.rename(columns={"ASR": "ASR (FN)"})
    df.to_csv(f"results/{key}_fn.csv", index=False)
